# this is for constructing the AI prompt. Do NOT run all without understanding the code below or talking with Noah

In [ ]:
#Set up
from pathlib import Path
import json
import os
import sys
import time
import warnings
import numpy as np
import logging

import pandas as pd
from IPython.display import display

rng = np.random.default_rng(seed=1312)
random_texts = rng.integers(1, 1061, size=35)

test_texts = pd.read_csv("full_text.csv", dtype={"id": str}).iloc[random_texts]
print(len(test_texts), "texts")  # Check: 50 texts
test_texts["president"].value_counts()

ModuleNotFoundError: No module named 'sentencepiece'

In [157]:
warnings.filterwarnings("ignore", category=FutureWarning)  # a harmless warning from one of the libraries

pd.set_option("display.max_colwidth", 300)  # show more of each text

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

Setup note: this course is tested with Python 3.13; you are running 3.14.7.
Python 3.14.7 | data=c:\Users\cgsta\Documents\Berkeley\Computing I\Pres speech\Miller-211A\ai_junk\compss211_data


In [158]:
print(COURSE_ROOT)

c:\Users\cgsta\Documents\Berkeley\Computing I\Pres speech\Miller-211A\ai_junk


In [159]:
test_texts['id'] = test_texts.index.astype(str)

In [160]:
test_texts['label'] = 'ignore'

### Ai set-up

In [161]:
# Supplied: connect to Gemini. The key is read from your .env file and never printed.
import logging

from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv(COURSE_ROOT / ".env")
key = os.environ.get("GEMINI_API_KEY", "")
if not key or key.startswith("paste-"):
    raise ValueError("Add GEMINI_API_KEY to your .env file, then run this cell again.")

client = genai.Client(api_key=key)
GEMINI_MODEL = "gemini-3.5-flash-lite"  # use the replacement model if bCourses announces one

# Hide a warning that the google-genai package prints on every call. It does not mean anything went wrong.
logging.getLogger("google_genai.types").setLevel(logging.ERROR)
print("Connected to Gemini.")

Connected to Gemini.


In [162]:
RETRY_CODES = {429, 503}


def ask_gemini(prompt, system, schema, max_tries=4):
    """Send one prompt with this system instruction and schema. Return the label and the reason. Try again after a temporary error."""
    wait = 10  # seconds to wait before the first retry
    for attempt in range(1, max_tries + 1):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system,
                    response_mime_type="application/json",
                    response_schema=schema,
                ),
            )
            time.sleep(6)  # wait between calls to stay under the free tier's limit per minute
            return json.loads(response.text)
        except errors.APIError as error:
            if error.code not in RETRY_CODES or attempt == max_tries:
                raise  # stop and show the error
            print(f"Error {error.code}. Waiting {wait} seconds, then trying again.")
            time.sleep(wait)
            wait = wait * 2  # wait longer each time: 10, 20, 40 seconds


def label_texts(texts, system, schema, path):
    """Ask Gemini about every text in texts. Save each answer to path right away."""
    # Texts that are already in the file were labeled in an earlier run, so skip them.
    done = set(pd.read_csv(path, dtype={"id": str})["id"]) if path.exists() else set()
    for text in texts.itertuples():
        if text.id in done:
            continue
        reply = ask_gemini(f"Text: {text.text}", system, schema)
        row = pd.DataFrame([{"id": text.id, "label": text.label, **reply, "model": GEMINI_MODEL}])
        # mode="a" adds one line to the end of the file. Write the column names only if the file is new.
        row.to_csv(path, mode="a", header=not path.exists(), index=False)
        done.add(text.id)
        print(len(done), "of", len(texts), "done")
    results = pd.read_csv(path, dtype={"id": str, "label": str, "llm_label": str})
    return results[results["id"].isin(texts["id"])]

In [163]:
PROJECT_SYSTEM = """You label texts from preidental speeches. One text is one row within the [text].

Give the text one label:
- welcoming: the text expresses a positive attitude towards immigration and immigrants/refugees, emphasizes American openness towards migrants, or calls for humanitarian support for migrants/refugees.
- restrictive: the text expresses a negative attitude towards immigration and immigrants/refugees; emphasizes border security, legal limits, deportation, or national security concerns regarding immigration; or calls for stricter immigration policies.
- neutral: the text does not express a clear positive or negative attitude towards immigration and immigrants/refugees; it is neither particularly welcoming nor restrictive.
- unclear: the text does not contain enough context to determine the attitude towards immigration and immigrants/refugees.
- ⁠not relevant: the text is functionally not related to immigration or immigrants/refugees.

Also give a reason of one short sentence.""" 
PROJECT_LABELS = ["welcoming", "restrictive",'neutral', "unclear", 'not relevant']

assert "..." not in PROJECT_SYSTEM and "..." not in PROJECT_LABELS, "Fill in PROJECT_SYSTEM and PROJECT_LABELS."
PROJECT_SCHEMA = {
    "type": "object",
    "properties": {
        "llm_label": {"type": "string", "enum": [str(label) for label in PROJECT_LABELS]},  # labels must be text, so 1 becomes "1"
        "llm_reason": {"type": "string"},
    },
    "required": ["llm_label", "llm_reason"],
}

In [164]:
write_texts = test_texts
score_texts = pd.read_csv('full_text.csv')
print(len(write_texts), "texts to write the prompt on;", len(score_texts), "to score it on")  # Check: 20 and 30

35 texts to write the prompt on; 1059 to score it on


In [165]:
first_results = label_texts(write_texts, PROJECT_SYSTEM, PROJECT_SCHEMA, GENERATED_DIR / "full_text.csv")
merged_results = write_texts.merge(first_results[['id', 'llm_label', 'llm_reason']], on='id')
merged_results[['president', 'year', 'title', 'speech_type', 'llm_label', 'llm_reason']].head()

,president,year,title,speech_type,llm_label,llm_reason
0,Andrew Jackson,1831,"February 22, 1831: Message Regarding Indian Relations",Message,not relavent,"The text discusses federal jurisdiction, state laws, and historical relations regarding Native American tribes, which falls under Indian affairs rather than immigration."
1,Andrew Jackson,1831,"February 22, 1831: Message Regarding Indian Relations",Message,not relavent,"The text discusses federal jurisdiction, state sovereignty, and Indian affairs, which is legally distinct from modern international immigration."
2,Andrew Jackson,1831,"February 22, 1831: Message Regarding Indian Relations",Message,not relavent,"The text discusses the jurisdictional rights of states over Native American tribes and Indian affairs, which is legally distinct from modern international immigration."
3,Chester A. Arthur,1884,"December 10, 1884: Message Regarding Central American Canal",Message,not relavent,The text discusses the construction of an interoceanic canal in Nicaragua and is not about immigration or immigrants.
4,Chester A. Arthur,1884,"December 10, 1884: Message Regarding Central American Canal",Message,not relavent,The text discusses the construction of an interoceanic canal in Nicaragua rather than immigration.


In [166]:
merged_results['llm_label'].value_counts()

llm_label
not relavent    33
welcoming        6
restrictive      2
neutral          1
Name: count, dtype: int64

In [167]:
merged_results['llm_label'] = merged_results['llm_label'].replace('not relavent', 'not relevant')

In [168]:
merged_results.to_csv("test_text.csv", index=False)

### Comparing results with hand-validation

In [169]:
score = pd.read_csv('HandValdatedList.csv')
score.head()

,id,hand_label,Reasoning
0,22,Not Relevant,"Racist man lowkey but mainly focuses on Native Americans and while it metioned emigration, it mostly talked about Native Americans moving West."
1,200,Not Relevant,Some water canal deal With Nicaragua
2,167,Not Relevant,Air Force thing?
3,23,Not Relevant,A lot of domestic and foreign policiy and relational things
4,836,not relevant,"A lot of economical stuff, also FUCK ROLAND RAEGAN"


In [170]:
score['hand_label'] = score['hand_label'].str.lower()

In [171]:
score.shape

(34, 3)

In [172]:
merged_results.shape

(42, 17)

In [173]:
merged_results['id'] = merged_results['id'].astype(int)

In [174]:
comparison = merged_results.merge(score, on='id', how='left')

In [175]:
comparison.duplicated().sum()

np.int64(4)

In [176]:
comparison = comparison.drop_duplicates()

In [177]:
print(comparison['hand_label'].value_counts())
print(comparison['llm_label'].value_counts())

hand_label
not relevant    32
welcoming        5
unclear          3
restrictive      1
Name: count, dtype: int64
llm_label
not relevant    33
welcoming        6
restrictive      2
neutral          1
Name: count, dtype: int64


In [178]:
comparison[comparison['hand_label'] != comparison['llm_label']]

,president,year,title,speech_type,word_count,text,war,peace,freedom,slavery,economy,taxes,immigration,id,label,llm_label,llm_reason,hand_label,Reasoning
15,John F. Kennedy,1961,"July 25, 1961: Report on the Berlin Crisis",Other,4093,"Good evening: Seven weeks ago tonight I returned from Europe to report on my meeting with Premier Khrushchev and the others. His grim warnings about the future of the world, his aide memoire on Berlin, his subsequent speeches and threats which he and his agents have launched, and the increase in...",10,19,17,0,2,3,0,661,ignore,not relevant,"The speech is primarily about the Berlin crisis, Cold War tensions, and military defense preparations rather than immigration.",unclear,"Talks about refugees but in the context of like East/West Germany and the Iron Curtain and the Berlin Wall, so it's not about the United States"
21,Ronald Reagan,1988,"May 31, 1988: Address at Moscow State University",Address / Speech,7140,"<strong>President Reagan:</strong> Thank you, Rector Logunov, and I want to thank all of you very much for a very warm welcome. It's a great pleasure to be here at Moscow State University, and I want to thank you all for turning out. I know you must be very busy this week, studying and taking yo...",8,6,24,1,4,0,0,877,ignore,welcoming,"The text emphasizes that America is a nation built on immigration and diversity, where anyone from around the world can come and become an American.",unclear,Has a mention of emigration but the context confuses me?
26,Lyndon B. Johnson,1963,"November 28, 1963: Thanksgiving Message",Message,2051,"My fellow Americans: On yesterday I went before the Congress to speak for the first time as President of the United States. Tonight, on this Thanksgiving, I come before you to ask your help, to ask your strength, to ask your prayers that God may guard this Republic and guide my every labor. All...",0,4,2,0,0,0,0,714,ignore,not relevant,"The speech is a Thanksgiving address focusing on national unity and mourning the assassination of President John F. Kennedy, with no mention of immigration.",NaN,NaN
37,Grover Cleveland,1893,"December 4, 1893: First Annual Message (Second Term)",State of the Union,12082,To the Congress of the United States: The constitutional duty which requires the President from time to time to give to the Congress information of the state of the Union and recommend to their consideration such measures as he shall judge necessary and expedient is fittingly entered upon by com...,12,0,1,0,6,3,1,448,ignore,restrictive,The text discusses immigration statistics detailing the number of immigrants not permitted to land or returned as public charges under the limitations of the law.,welcoming,"Talks about letting 400K migrants into the country, mentions not letting in 1500 people in but like very small percentage"
44,James K. Polk,1846,"August 5, 1846: Message Requesting the Creation of an Oregon Territory",Message,344,"To the Senate and House of Representatives of the United States:I communicate herewith a copy of a convention for the settlement and adjustment of the Oregon question, which was concluded in this city on the 15th day of June last between the United States and Her Britannic Majesty. This conventi...",0,0,0,0,0,0,0,535,ignore,welcoming,The text encourages settlers to move to the Oregon territory and recommends providing them with liberal land grants to aid their journey.,not relevant,Talks about Native American land
45,Andrew Johnson,1866,"December 3, 1866: Second Annual Message",State of the Union,7068,"Fellow-Citizens of the Senate and House of Representatives:After a brief interval the Congress of the United States resumes its annual legislative labors. An all-wise and merciful Providence has abated the pestilence which visited our shores, leaving its calamitous traces upon some portions of o...",16,6,0,2,1,1,0,55,ignore,neutral,"The text discusses general land policies, emigration, and naturalization laws without expressing a strong positi

In [179]:
comparison['hand_label'] = comparison['hand_label'].replace('unclear', pd.NA)

In [180]:
cleaned_comparison = comparison.dropna(subset=['hand_label'])

In [182]:
cleaned_comparison[cleaned_comparison['hand_label'] != cleaned_comparison['llm_label']]

,president,year,title,speech_type,word_count,text,war,peace,freedom,slavery,economy,taxes,immigration,id,label,llm_label,llm_reason,hand_label,Reasoning
37,Grover Cleveland,1893,"December 4, 1893: First Annual Message (Second Term)",State of the Union,12082,To the Congress of the United States: The constitutional duty which requires the President from time to time to give to the Congress information of the state of the Union and recommend to their consideration such measures as he shall judge necessary and expedient is fittingly entered upon by com...,12,0,1,0,6,3,1,448,ignore,restrictive,The text discusses immigration statistics detailing the number of immigrants not permitted to land or returned as public charges under the limitations of the law.,welcoming,"Talks about letting 400K migrants into the country, mentions not letting in 1500 people in but like very small percentage"
44,James K. Polk,1846,"August 5, 1846: Message Requesting the Creation of an Oregon Territory",Message,344,"To the Senate and House of Representatives of the United States:I communicate herewith a copy of a convention for the settlement and adjustment of the Oregon question, which was concluded in this city on the 15th day of June last between the United States and Her Britannic Majesty. This conventi...",0,0,0,0,0,0,0,535,ignore,welcoming,The text encourages settlers to move to the Oregon territory and recommends providing them with liberal land grants to aid their journey.,not relevant,Talks about Native American land


In [185]:
(cleaned_comparison['hand_label'] != cleaned_comparison['llm_label']).mean().sum()

np.float64(0.05263157894736842)

## Token estimater

In [190]:
from google.genai.local_tokenizer import LocalTokenizer

In [194]:
full_text = pd.read_csv("full_text.csv", dtype={"id": int})

In [199]:
collected_text = ""
for text in full_text['text']:
    collected_text += text + " "

In [200]:
tokenizer = LocalTokenizer(model_name="gemini-3-flash-preview")
result = tokenizer.count_tokens(collected_text)

print(f"Estimated local tokens: {result.total_tokens}")

Estimated local tokens: 4966852
